# 02 OPERA DISP-S1 Product Search

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roymustang11/InSAR-Benchmark-Lab/blob/main/notebooks/02_hyp3_or_opera_to_timeseries.ipynb)

This notebook starts the first real data path for the Central Valley subsidence benchmark: discovering OPERA Sentinel-1 surface displacement products and inspecting product metadata without downloading large science files into the repository.

The target collection is `OPERA_L3_DISP-S1_V1`. The notebook performs metadata-first discovery, builds a small inventory table, and includes an authenticated opt-in path for inspecting Zarr-reference metadata.


## Scope

This notebook does three things:

1. loads the shared Central Valley study-area configuration,
2. defines an OPERA DISP-S1 metadata search using NASA Earthdata / CMR through `earthaccess`,
3. converts search results into a small inventory table suitable for later download, streaming, or product comparison,
4. identifies lightweight product-structure metadata links for a first product inspection.

It does **not** commit or download large NetCDF, HDF5, GeoTIFF, or Zarr products. Large products belong under ignored local directories such as `data/raw/` or `data/external/`. Authenticated metadata inspection is opt-in because OPERA product files are protected by Earthdata Login.


## Source Notes

- OPERA DISP-S1 is the OPERA Level-3 surface displacement product derived from Sentinel-1. NASA Earthdata describes it as useful for deformation applications including subsidence, tectonics, and landslides.
- `earthaccess` searches NASA's Common Metadata Repository and can download or stream NASA Earthdata products after authentication.
- This notebook starts with metadata search only. Download/authentication will be added after the first target products are selected.

Useful references:

- [NASA Earthdata OPERA project](https://www.earthdata.nasa.gov/data/projects/opera)
- [OPERA products documentation](https://nasa-opera.github.io/docs/products/)
- [earthaccess search guide](https://earthaccess.readthedocs.io/en/latest/user_guide/search/)


## Setup

The notebook is safe to execute by default. `RUN_LIVE_SEARCH` is set to `False` so automated checks and first-time readers do not depend on network access. `RUN_AUTHENTICATED_INSPECTION` is also `False` because inspecting protected OPERA metadata links requires Earthdata Login credentials.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/roymustang11/InSAR-Benchmark-Lab.git"
RUN_LIVE_SEARCH = False
RUN_AUTHENTICATED_INSPECTION = False

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    project_root = Path("/content/InSAR-Benchmark-Lab")
    if not project_root.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(project_root)])
else:
    cwd = Path.cwd().resolve()
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

config_path = project_root / "configs" / "central_valley_subsidence.yml"
inventory_dir = project_root / "data" / "interim" / "opera_disp_s1"
inventory_path = inventory_dir / "central_valley_search_inventory.csv"

print(f"Project root: {project_root}")
print(f"Live Earthdata search enabled: {RUN_LIVE_SEARCH}")
print(f"Authenticated metadata inspection enabled: {RUN_AUTHENTICATED_INSPECTION}")


## Load Study-Area Configuration


In [ ]:
from insar_benchmark_lab.config import load_study_area_config

config = load_study_area_config(config_path)
bbox = config.region
search_bounding_box = (bbox["west"], bbox["south"], bbox["east"], bbox["north"])
search_temporal = (config.time_window["start"], config.time_window["end"])

print(config.name)
print("Bounding box:", search_bounding_box)
print("Temporal window:", search_temporal)


## Define OPERA DISP-S1 Search

The Earthdata query is deliberately small. It asks for a limited number of granules intersecting the Central Valley bounding box and configured time window.


In [ ]:
from insar_benchmark_lab.opera import OPERA_DISP_S1_SHORT_NAME

MAX_GRANULES = 10

search_kwargs = {
    "short_name": OPERA_DISP_S1_SHORT_NAME,
    "bounding_box": search_bounding_box,
    "temporal": search_temporal,
    "count": MAX_GRANULES,
}

search_kwargs


## Run Metadata Search

Set `RUN_LIVE_SEARCH = True` in the setup cell to query NASA Earthdata. Metadata search should not require downloading large products. Download cells will be added only after the first target granules are selected.


In [ ]:
import pandas as pd

if RUN_LIVE_SEARCH:
    import earthaccess

    results = earthaccess.search_data(**search_kwargs)
else:
    results = []
    print("RUN_LIVE_SEARCH is False; skipping earthaccess.search_data.")

print(f"Granules returned: {len(results)}")


## Build Inventory Table

The inventory is intentionally small and text-based. It can be committed only if it contains metadata and no large product payloads. By default this notebook does not write anything because live search is disabled.


In [ ]:
from insar_benchmark_lab.opera import granule_to_inventory_record

inventory = pd.DataFrame([granule_to_inventory_record(granule) for granule in results])
inventory


## Select First Product For Inspection

When live search returns results, the inventory table separates the large NetCDF product from lightweight Zarr-reference metadata links. The Zarr-reference link is the preferred first inspection target because it can reveal variable structure without downloading the full product payload.


In [ ]:
if not inventory.empty:
    first_product = inventory.iloc[0].to_dict()
    print("First granule:", first_product.get("granule_id"))
    print("Frame:", first_product.get("frame_id"))
    print("Reference/secondary:", first_product.get("reference_datetime"), first_product.get("secondary_datetime"))
    print("NetCDF link:", first_product.get("netcdf_link"))
    print("Zarr-reference link:", first_product.get("zarr_reference_link"))
else:
    first_product = {}
    print("No live inventory available. Set RUN_LIVE_SEARCH = True to select a product.")


## Authenticated Product-Structure Inspection

The OPERA DISP-S1 data links returned by Earthdata are protected. To inspect the first product's Zarr-reference metadata, set both flags in the setup cell:

```python
RUN_LIVE_SEARCH = True
RUN_AUTHENTICATED_INSPECTION = True
```

Then rerun the notebook. `earthaccess.login()` will use a `.netrc` file, environment variables, or an interactive login flow depending on your environment. The cell stores only a small metadata JSON under `data/interim/`, which is ignored by Git.


In [ ]:
import gzip
import json
from insar_benchmark_lab.opera import extract_zarr_reference_variables

zarr_reference = first_product.get("zarr_reference_link")
zarr_reference_json = None
zarr_variables = []

if RUN_AUTHENTICATED_INSPECTION and zarr_reference:
    import earthaccess

    earthaccess.login()
    metadata_path = inventory_dir / "first_product_zarr_reference.json"
    inventory_dir.mkdir(parents=True, exist_ok=True)

    # The authenticated session configures Earthdata cookies for protected links.
    session = earthaccess.get_requests_https_session()
    response = session.get(zarr_reference, timeout=60)
    response.raise_for_status()
    payload = gzip.decompress(response.content) if zarr_reference.endswith(".gz") else response.content
    zarr_reference_json = json.loads(payload)
    metadata_path.write_text(json.dumps(zarr_reference_json, indent=2), encoding="utf-8")
    zarr_variables = extract_zarr_reference_variables(zarr_reference_json)
    print(f"Wrote metadata JSON: {metadata_path}")
    print("Top-level arrays:")
    for variable in zarr_variables:
        print("-", variable)
else:
    print("Authenticated inspection skipped. Enable RUN_AUTHENTICATED_INSPECTION after confirming Earthdata login is configured.")


In [ ]:
if RUN_LIVE_SEARCH and not inventory.empty:
    inventory_dir.mkdir(parents=True, exist_ok=True)
    inventory.to_csv(inventory_path, index=False)
    print(f"Wrote metadata inventory: {inventory_path}")
else:
    print("No inventory written. Enable RUN_LIVE_SEARCH and rerun after checking the search parameters.")


## Next Steps

After confirming the OPERA DISP-S1 search and authenticated metadata inspection return suitable Central Valley granules:

1. inspect the first product's Zarr-reference metadata without committing the product payload,
2. identify displacement, temporal coherence, acquisition-date, and spatial-reference arrays,
3. add a small extraction notebook section for one station or pixel-level time series,
4. compare that time series with GNSS in `03_mintpy_timeseries_validation.ipynb`.

If OPERA coverage or authentication blocks the first experiment, use ASF HyP3 as the fallback data path and keep the same inventory-table pattern.
